# Copy Fan/Slider/Valve WAVs Into Pump-Style Structure

This notebook scans `datasets/fan`, `datasets/slider`, and `datasets/valve`, then copies files into
pump-style flat folders under `datasets/`:

- `+0db-<machine>-normal`
- `+0db-<machine>-abnormal`
- `+6db-<machine>-normal`
- `+6db-<machine>-abnormal`
- `-6db-<machine>-normal`
- `-6db-<machine>-abnormal`

Safety behavior:

- Copy-only (`shutil.copy2`)
- Source files are never moved or deleted
- Existing destination files are never overwritten
- Preview first, copy only when `PROCESS = True`


In [4]:
from pathlib import Path
import re
import shutil

import pandas as pd
from IPython.display import Markdown, display

PROCESS = True
MACHINES = ('fan', 'slider', 'valve')
DISPLAY_LIMIT = 200

FILENAME_RE = re.compile(
    r'(?P<db>[+-]\d+)-(?P<machine>[A-Za-z0-9_-]+)-(?P<id>\d+)-(?P<rec>ab|nm)-(?P<tail>[A-Za-z0-9]+)\.wav$',
    re.IGNORECASE,
)
REC_TO_LABEL = {'nm': 'normal', 'ab': 'abnormal'}


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'datasets').exists() and (candidate / 'utilities').exists():
            return candidate
    raise RuntimeError('Could not infer repo root. Open notebook from inside this repo.')


def normalise_machine(machine: str) -> str:
    return re.sub(r'[^a-z0-9]+', '-', machine.strip().lower()).strip('-')


def parse_filename(wav_path: Path) -> dict:
    m = FILENAME_RE.fullmatch(wav_path.name)
    if not m:
        raise ValueError(f'Unexpected filename format: {wav_path.name}')

    parts = m.groupdict()
    db = f"{int(parts['db']):+d}"
    machine = normalise_machine(parts['machine'])
    rec = parts['rec'].lower()
    if rec not in REC_TO_LABEL:
        raise ValueError(f"Unexpected rec token '{rec}' in {wav_path.name}")

    return {
        'db': db,
        'machine': machine,
        'recording_type': REC_TO_LABEL[rec],
    }


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATASETS_DIR = REPO_ROOT / 'datasets'

display(Markdown(f'Using repo root: `{REPO_ROOT}`'))
print(f'PROCESS: {PROCESS}')
print(f'MACHINES: {MACHINES}')
print(f'DISPLAY_LIMIT: {DISPLAY_LIMIT}')


Using repo root: `/home/mitch/development/raccoon-ball`

PROCESS: True
MACHINES: ('fan', 'slider', 'valve')
DISPLAY_LIMIT: 200


In [5]:
rows = []
errors = []

for machine_dir_name in MACHINES:
    source_root = DATASETS_DIR / machine_dir_name
    if not source_root.exists():
        errors.append({
            'source_path': str(source_root),
            'error': f'Missing source directory: {source_root}',
        })
        continue

    wav_paths = sorted(source_root.rglob('*.wav'))
    for wav_path in wav_paths:
        try:
            parsed = parse_filename(wav_path)
            machine = parsed['machine']
            db = parsed['db']
            recording_type = parsed['recording_type']

            expected_machine = normalise_machine(machine_dir_name)
            if machine != expected_machine:
                raise ValueError(
                    f'Filename machine token {machine!r} does not match source directory {expected_machine!r}'
                )

            dest_dir = DATASETS_DIR / f'{db}db-{machine}-{recording_type}'
            dest_path = dest_dir / wav_path.name
            action = 'skip_exists' if dest_path.exists() else 'copy'

            rows.append({
                'source_path': str(wav_path),
                'destination_path': str(dest_path),
                'machine': machine,
                'db': db,
                'recording_type': recording_type,
                'action': action,
                'source_exists_now': wav_path.exists(),
                'dest_exists_now': dest_path.exists(),
            })
        except Exception as exc:
            errors.append({
                'source_path': str(wav_path),
                'error': f'{type(exc).__name__}: {exc}',
            })

PREVIEW_DF = pd.DataFrame(rows)
ERRORS_DF = pd.DataFrame(errors)

if PREVIEW_DF.empty:
    print('No wav files found for configured machine directories.')
else:
    summary = (
        PREVIEW_DF
        .groupby(['machine', 'db', 'recording_type', 'action'])
        .size()
        .reset_index(name='count')
        .sort_values(['machine', 'db', 'recording_type', 'action'])
    )
    display(summary)

    copy_count = int((PREVIEW_DF['action'] == 'copy').sum())
    skip_count = int((PREVIEW_DF['action'] == 'skip_exists').sum())
    print(f'Total files discovered: {len(PREVIEW_DF)}')
    print(f'Copy candidates: {copy_count}')
    print(f'Skipped because destination exists: {skip_count}')

    display(PREVIEW_DF.head(DISPLAY_LIMIT))

if not ERRORS_DF.empty:
    display(ERRORS_DF.head(DISPLAY_LIMIT))
    raise RuntimeError(f'Found {len(ERRORS_DF)} parsing/validation errors. Resolve before copying.')


,machine,db,recording_type,action,count
0,fan,+0,abnormal,copy,1475
1,fan,+0,normal,copy,4075
2,fan,+6,abnormal,copy,1475
3,fan,+6,normal,copy,4075
4,fan,-6,abnormal,copy,1475
5,fan,-6,normal,copy,4075
6,slider,+0,abnormal,copy,890
7,slider,+0,normal,copy,3204
8,slider,+6,abnormal,copy,890
9,slider,+6,normal,copy,3204


Total files discovered: 41442
Copy candidates: 41442
Skipped because destination exists: 0


,source_path,destination_path,machine,db,recording_type,action,source_exists_now,dest_exists_now
0,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
1,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
2,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
3,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
4,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
...,...,...,...,...,...,...,...,...
195,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
196,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
197,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False
198,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,fan,+0,abnormal,copy,True,False


In [6]:
if 'PREVIEW_DF' not in globals():
    raise RuntimeError('Run the preview cell first to build PREVIEW_DF.')

if 'ERRORS_DF' in globals() and not ERRORS_DF.empty:
    raise RuntimeError('Preview has errors. Fix those first; no files copied.')

if not PROCESS:
    print('PROCESS is False. No files were copied.')
else:
    candidates = PREVIEW_DF[PREVIEW_DF['action'] == 'copy'].copy()

    if candidates.empty:
        print('No copy candidates found. Nothing to do.')
    else:
        copied_rows = []

        for row in candidates.itertuples(index=False):
            src = Path(row.source_path)
            dst = Path(row.destination_path)

            if not src.exists():
                raise FileNotFoundError(f'Source disappeared before copy: {src}')

            if dst.exists():
                # Safety: never overwrite existing destination files.
                continue

            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

            copied_rows.append({
                'source_path': str(src),
                'destination_path': str(dst),
                'source_still_exists': src.exists(),
                'dest_exists_after_copy': dst.exists(),
            })

        RESULT_DF = pd.DataFrame(copied_rows)
        print(f'Copy complete. Files copied: {len(RESULT_DF)}')

        if not RESULT_DF.empty:
            if not bool(RESULT_DF['source_still_exists'].all()):
                raise RuntimeError('One or more sources are missing after copy. Expected copy-only behavior.')
            if not bool(RESULT_DF['dest_exists_after_copy'].all()):
                raise RuntimeError('One or more destination files are missing after copy.')

            display(RESULT_DF.head(DISPLAY_LIMIT))


Copy complete. Files copied: 41442


,source_path,destination_path,source_still_exists,dest_exists_after_copy
0,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
1,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
2,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
3,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
4,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
...,...,...,...,...
195,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
196,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
197,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
198,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/datasets/...,True,True
